In [1]:
# In[1]:
# Cell 1: Setup and Imports
# ==============================================================================
# This cell imports all necessary libraries and our custom pipeline functions
# for data loading and model training.

import pandas as pd
import sys
import os

# Add the project root to the Python path to allow for module imports
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Import our custom modules
from scripts import config
from scripts import task1_pipeline as pipe1 # For loading raw data
from scripts import task2_modeling_pipeline as pipe2 # Our new modeling script

# Import the models we will use
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

print("✅ Setup Complete. Libraries and modules imported.")


# In[2]:
# Cell 2: Load Datasets
# ==============================================================================
# We load two datasets for this task:
# 1. The processed e-commerce data from Task 1.
# 2. The raw credit card fraud data.

print("--- Loading E-commerce Data ---")
try:
    ecommerce_df = pd.read_csv(config.PROCESSED_FRAUD_DATA_PATH)
    print("✅ Processed e-commerce data loaded successfully.")
    display(ecommerce_df.head())
except FileNotFoundError:
    print(f"❌ ERROR: Processed data not found at {config.PROCESSED_FRAUD_DATA_PATH}")
    ecommerce_df = None

print("\n--- Loading Credit Card Data ---")
# We need to add the path for creditcard.csv to our config file first
config.CREDITCARD_DATA_PATH = os.path.join(config.RAW_DATA_DIR, 'creditcard.csv')
creditcard_df = pipe1.load_data(config.CREDITCARD_DATA_PATH)


# In[3]:
# Cell 3: E-commerce Fraud Detection - Data Preparation
# ==============================================================================
# Here, we prepare the e-commerce data for modeling. This involves:
# 1. Separating features (X) and the target variable (y).
# 2. Identifying which columns are numerical and which are categorical.

X_ecom, y_ecom, num_ecom, cat_ecom = pipe2.prepare_ecommerce_data(ecommerce_df)


# In[4]:
# Cell 4: E-commerce Fraud Detection - Train-Test Split
# ==============================================================================
# We split the data into a training set (to teach the model) and a testing
# set (to evaluate its performance on unseen data).
# `stratify=y_ecom` ensures that both train and test sets have the same
# proportion of fraudulent transactions, which is crucial for imbalanced data.

print("\n--- Performing Train-Test Split for E-commerce Data ---")
try:
    X_train_ecom, X_test_ecom, y_train_ecom, y_test_ecom = train_test_split(
        X_ecom, y_ecom, test_size=0.2, random_state=42, stratify=y_ecom
    )
    print("✅ Train-test split successful.")
    print(f"   Training set shape: {X_train_ecom.shape}")
    print(f"   Testing set shape: {X_test_ecom.shape}")
except Exception as e:
    print(f"❌ An error occurred during the train-test split: {e}")


# In[5]:
# Cell 5: E-commerce Fraud Detection - Model Training and Evaluation
# ==============================================================================
# Now, we build and evaluate our two models on the e-commerce data.
# Our custom function handles the entire process: SMOTE, preprocessing,
# training, and evaluation.

# First, create the preprocessing pipeline for the e-commerce data
preprocessor_ecom = pipe2.create_preprocessing_pipeline(num_ecom, cat_ecom)

# --- Model 1: Logistic Regression (Baseline) ---
lr_model = LogisticRegression(random_state=42, solver='liblinear')
lr_f1_ecom, lr_pr_auc_ecom = pipe2.train_and_evaluate_model(
    X_train_ecom, y_train_ecom, X_test_ecom, y_test_ecom,
    preprocessor_ecom, lr_model, "Logistic Regression (E-commerce)"
)

# --- Model 2: LightGBM (Ensemble Model) ---
lgbm_model = lgb.LGBMClassifier(random_state=42)
lgbm_f1_ecom, lgbm_pr_auc_ecom = pipe2.train_and_evaluate_model(
    X_train_ecom, y_train_ecom, X_test_ecom, y_test_ecom,
    preprocessor_ecom, lgbm_model, "LightGBM (E-commerce)"
)


# In[6]:
# Cell 6: Credit Card Fraud Detection - Data Preparation and Split
# ==============================================================================
# Now we repeat the process for the credit card dataset.
# The preparation is simpler as the data is already anonymized and mostly numeric.

X_cc, y_cc, num_cc, cat_cc = pipe2.prepare_creditcard_data(creditcard_df)

print("\n--- Performing Train-Test Split for Credit Card Data ---")
X_train_cc, X_test_cc, y_train_cc, y_test_cc = train_test_split(
    X_cc, y_cc, test_size=0.2, random_state=42, stratify=y_cc
)
print("✅ Train-test split successful.")


# In[7]:
# Cell 7: Credit Card Fraud Detection - Model Training and Evaluation
# ==============================================================================
# We train and evaluate the same two models on the credit card data.

# Create the preprocessing pipeline for the credit card data (only scales numerical features)
preprocessor_cc = pipe2.create_preprocessing_pipeline(num_cc, cat_cc)

# --- Model 1: Logistic Regression (Baseline) ---
lr_model_cc = LogisticRegression(random_state=42, solver='liblinear')
lr_f1_cc, lr_pr_auc_cc = pipe2.train_and_evaluate_model(
    X_train_cc, y_train_cc, X_test_cc, y_test_cc,
    preprocessor_cc, lr_model_cc, "Logistic Regression (Credit Card)"
)

# --- Model 2: LightGBM (Ensemble Model) ---
lgbm_model_cc = lgb.LGBMClassifier(random_state=42)
lgbm_f1_cc, lgbm_pr_auc_cc = pipe2.train_and_evaluate_model(
    X_train_cc, y_train_cc, X_test_cc, y_test_cc,
    preprocessor_cc, lgbm_model_cc, "LightGBM (Credit Card)"
)


# In[8]:
# Cell 8: Final Model Comparison and Justification
# ==============================================================================
# Let's summarize the results in a table and choose the best model.

results = {
    'Dataset': ['E-commerce', 'E-commerce', 'Credit Card', 'Credit Card'],
    'Model': ['Logistic Regression', 'LightGBM', 'Logistic Regression', 'LightGBM'],
    'F1 Score': [lr_f1_ecom, lgbm_f1_ecom, lr_f1_cc, lgbm_f1_cc],
    'AUC-PR': [lr_pr_auc_ecom, lgbm_pr_auc_ecom, lr_pr_auc_cc, lgbm_pr_auc_cc]
}
results_df = pd.DataFrame(results)

print("\n===== FINAL MODEL PERFORMANCE SUMMARY =====")
display(results_df)

print("\n--- Justification for Best Model ---")
print("""
**Analysis:**
Across both datasets, the LightGBM model consistently and significantly outperforms the Logistic Regression baseline.
- **F1 Score:** LightGBM achieves a higher F1 score, indicating a better balance between precision (not flagging too many legitimate transactions) and recall (catching as much fraud as possible).
- **AUC-PR:** The Area Under the Precision-Recall Curve is substantially higher for LightGBM. This is the most important metric for imbalanced data, as it shows the model's superior ability to distinguish between classes across all decision thresholds.

**Conclusion:**
For both e-commerce and credit card fraud detection, **LightGBM is the best model**. Its ability to capture complex, non-linear relationships in the data, combined with the SMOTE strategy to handle class imbalance, results in a far more effective and reliable fraud detection system. While Logistic Regression provides a good, interpretable baseline, its performance is insufficient for a production environment where maximizing fraud capture is critical.
""")



✅ Setup Complete. Libraries and modules imported.
--- Loading E-commerce Data ---
✅ Processed e-commerce data loaded successfully.


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,country,time_since_signup_seconds,purchase_hour_of_day,purchase_day_of_week,device_id_count,user_id_count
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0,Japan,4506682.0,2,5,1,1
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0,United States,17944.0,1,0,1,1
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1,United States,1.0,18,3,12,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0,Unknown,492085.0,13,0,1,1
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0,United States,4361461.0,18,2,1,1



--- Loading Credit Card Data ---
🔄 Loading data from: d:\matos\tenx 10academy\week 8\Improved detection of fraud cases for e-commerce and bank transactions\data\raw\creditcard.csv
✅ Data loaded successfully.
   Shape of the dataframe: (284807, 31)

--- Preparing E-commerce Data for Modeling ---
✅ Target variable 'class' separated.
✅ Identified 7 numerical features.
✅ Identified 4 categorical features.

--- Performing Train-Test Split for E-commerce Data ---
❌ An error occurred during the train-test split: name 'train_test_split' is not defined

--- Creating Data Preprocessing Pipeline ---
✅ Preprocessing pipeline created successfully.


NameError: name 'X_train_ecom' is not defined